# 6. Supervisor

*Using Microsoft Semantic Kernel (Agent Framework)*

Introduces the supervisor pattern where a central agent delegates tasks to specialized worker agents. The supervisor coordinates multiple agents, each with specific tools and capabilities, to accomplish complex tasks.

In [ ]:
import os
from typing import Annotated
from dotenv import load_dotenv
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

load_dotenv()

kernel = sk.Kernel()
service_id = "chat-gpt"
kernel.add_service(
    AzureChatCompletion(
        service_id=service_id,
        deployment_name="gpt-4o-mini",
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    )
)

In [ ]:
# Define worker plugins
class ResearchPlugin:
    @kernel_function(name="research", description="Research a topic and provide information")
    def research(self, topic: Annotated[str, "Topic to research"]) -> str:
        return f"Research results for {topic}: AI agents are autonomous systems that perceive, reason, and act."

class WriterPlugin:
    @kernel_function(name="write", description="Write content based on research")
    def write(self, content: Annotated[str, "Content to write about"]) -> str:
        return f"Article draft: {content} AI agents represent a significant advancement in automation."

class EditorPlugin:
    @kernel_function(name="edit", description="Edit and improve content")
    def edit(self, draft: Annotated[str, "Draft to edit"]) -> str:
        return f"Edited version: {draft} [Improved clarity and structure]"

# Add worker plugins
kernel.add_plugin(ResearchPlugin(), plugin_name="researcher")
kernel.add_plugin(WriterPlugin(), plugin_name="writer")
kernel.add_plugin(EditorPlugin(), plugin_name="editor")

In [ ]:
async def supervisor_workflow(task: str) -> str:
    """Supervisor coordinates worker agents to complete a task."""
    
    chat_history = ChatHistory()
    chat_history.add_system_message(
        "You are a supervisor that coordinates research, writing, and editing tasks. "
        "Use the available tools in sequence: first research, then write, then edit."
    )
    chat_history.add_user_message(task)
    
    execution_settings = AzureChatPromptExecutionSettings(
        service_id=service_id,
        function_choice_behavior=FunctionChoiceBehavior.Auto(),
    )
    
    chat_service = kernel.get_service(service_id)
    
    # The supervisor automatically coordinates the tools
    response = await chat_service.get_chat_message_content(
        chat_history=chat_history,
        settings=execution_settings,
        kernel=kernel,
    )
    
    return str(response)

In [ ]:
# Test supervisor pattern
result = await supervisor_workflow("Create an article about AI agents")
print("Supervisor Result:")
print(result)

**Note:** The supervisor pattern in Semantic Kernel uses automatic function calling to coordinate multiple specialized plugins. The LLM acts as the supervisor, intelligently selecting and orchestrating the worker tools.